# Exp14: head-output patching across languages (same names)
Question: heads are shared (Exp13). Do they write in a **language-agnostic output format**, or into language-specific subspaces?

Design: matched prompts. Same names, same name order, same template index (templates are aligned translations), English vs French / Yoruba (English names, so the answer token is identical).
Patch the top-K IOI heads' outputs (pre-`o_proj`) from the EN run into the FR/YO run, position-matched, and read the logit-diff.

Anchors (all on the same target prompts):
- **clean**: no intervention.
- **ablated**: same heads zeroed at the same positions.
- **cross**: heads patched EN -> tgt.
- **floor**: heads patched from a *different template in the same language* (tgt template t <- tgt template t+1). If this isn't near clean, patching at these positions is too brittle to interpret.
- **random**: random heads patched EN -> tgt.

`compat = (x - ablated) / (clean - ablated)`: 1 = interchangeable, 0 = as bad as removing the heads.
Read: cross ~ floor ~ 1 -> shared output format. cross ~ random ~ 0 while floor ~ 1 -> language-specific output format (separate subspaces, or language-specific downstream readers; this test cannot tell which). Between -> partial.
Position modes: `end` (END only) and `named` (IO, S2, END, each matched to its own source position).

Head rankings are reused from `exp13_results/<model>.json` when present (full runs only), otherwise scanned here. **Run SMOKE=True first.**

In [1]:
import os
GPU_ID = "0"   # your sentinel; None = all visible GPUs
if GPU_ID is not None: os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID   # before torch import
import json, gc, random
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SMOKE = False
HF_TOKEN = os.environ.get("HF_TOKEN")
MODEL_ROOT = "/home/models"      # your earlier output printed /home/models/...; change to /home2/models if that is the real mount
MODELS = ["/home/models/gemma-2-9b-it", "/home/models/gemma-2-27b-it",
          "google/gemma-2-2b", "/home/models/gemma-2-9b"]
LANGS = ["en", "fr", "yo"]
POSMODES = {"end": ["END"], "named": ["IO", "S2", "END"]}
GATE_ACC = 0.75
MIN_ABL = 0.3     # ablation must remove at least this fraction of clean logit-diff for compat to count
RESULTS_DIR = "exp14_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

if SMOKE:
    MODELS = MODELS[:1]; N_PAIRS = 10; SEEDS = [0]; K_LIST = [5, 10]; SCAN_PER_T = 2
else:
    N_PAIRS = 48; SEEDS = [0, 1, 2]; K_LIST = [5, 10]; SCAN_PER_T = 4
assert torch.cuda.is_available(), "no CUDA visible: check GPU_ID"
props = torch.cuda.get_device_properties(0)
print("SMOKE" if SMOKE else "FULL", [os.path.basename(m) for m in MODELS], props.name, round(props.total_memory/2**30), "GiB")

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FULL ['gemma-2-9b-it', 'gemma-2-27b-it', 'gemma-2-2b', 'gemma-2-9b'] NVIDIA H200 NVL 140 GiB


In [2]:
TEMPLATES = {
 "en": ["When {N1} and {N2} went to the store, {S} gave a drink to",
        "Then, {N1} and {N2} had a lot of fun at the school. {S} gave a ball to",
        "After {N1} and {N2} went to the park, {S} handed a book to",
        "Yesterday {N1} and {N2} visited the office, and {S} sent a letter to",
        "While {N1} and {N2} were at the market, {S} offered some fruit to",
        "Today {N1} and {N2} met in the garden. Afterwards, {S} showed the photos to"],
 "fr": ["Quand {N1} et {N2} sont allés au marché, {S} a donné une pomme à",
        "Ensuite, {N1} et {N2} se sont bien amusés à l'école. {S} a donné un ballon à",
        "Après que {N1} et {N2} sont allés au parc, {S} a prêté un livre à",
        "Hier, {N1} et {N2} ont visité le bureau, et {S} a envoyé une lettre à",
        "Pendant que {N1} et {N2} étaient au marché, {S} a offert des fruits à",
        "Aujourd'hui, {N1} et {N2} se sont retrouvés dans le jardin. Ensuite, {S} a montré les photos à"],
 "yo": ["Nígbà tí {N1} àti {N2} lọ sí ọjà, {S} ra ẹ̀bùn kan fún",
        "Lẹ́yìn náà, {N1} àti {N2} gbádùn ní ilé-ẹ̀kọ́. {S} fi bọ́ọ̀lù kan fún",
        "Lẹ́yìn tí {N1} àti {N2} lọ sí ọgbà, {S} yá ìwé kan fún",
        "Lánàá {N1} àti {N2} ṣàbẹ̀wò sí ọ́fíìsì, {S} sì fi lẹ́tà kan ranṣẹ́ sí",
        "Nígbà tí {N1} àti {N2} wà ní ọjà, {S} mú èso wá fún",
        "Lónìí {N1} àti {N2} pàdé ní ọgbà. Lẹ́yìn náà, {S} fi àwòrán hàn fún"],
}
NAMES = {
 "en": "James John Robert Michael William David Richard Thomas Mary Patricia Jennifer Linda Elizabeth Barbara Susan Jessica Sarah Karen Daniel Mark Paul Steven Kevin Brian George Anna Laura Emma Alice Julia Peter Henry Frank Kate Lucy Ruth Helen Carol Diane Grace Rose Edward Andrew Joseph Charles".split(),
 "fr": "Jean Pierre Louis Jacques Michel Henri Marie Claire Sophie Camille Lucie Chloé Hugo Lucas Antoine Nicolas Alain Luc Éric Anne Isabelle Nathalie Sylvie Céline Julien Mathieu Olivier Bruno Léa Manon Étienne Gérard Vincent Émile Alexandre Margot Inès Mathilde Clémence Aurélie Baptiste Guillaume".split(),
 "yo": "Ayo Tunde Femi Bola Kemi Dayo Tola Segun Yemi Wale Funmi Ade Bisi Folake Kunle Lola Sade Tayo Seyi Bimpe Dupe Yinka Kola Ladi Remi Toyin Jide Gbenga Ife Tobi Temi Wumi Niyi Shola Sola Lekan Dele Kayode Bukola Ronke Shade Bayo Biodun Taiwo Kehinde Lanre".split(),
}
NAMES["fr"] = [n for n in NAMES["fr"] if n not in set(NAMES["en"])]   # keep lexically disjoint from EN

In [3]:
NAMES["yo"] = list(dict.fromkeys(NAMES["yo"] + (
 "Abiola Adebayo Adewale Akin Akinwale Ayodele Babajide Babatunde Bolu Busola Damilola Debo Deji Dotun Efe Eniola "
 "Folu Gbemi Idowu Ige Jumoke Kike Lade Mide Muyiwa Nike Niran Olu Omolara Rotimi Sanmi Simi Tade Tope Tosin Wunmi Yetunde Yomi "
 "Adeola Femisoye Ibukun Jola Lara Moji Ola Tomi Yewande Zainab").split()))

In [4]:
def single_token_names(tok, names):
    out, nid, seen = [], {}, set()
    for n in names:
        ids = tok.encode("a " + n, add_special_tokens=False)
        if len(ids) == 2 and ids[1] not in seen:
            out.append(n); nid[n] = ids[1]; seen.add(ids[1])
    return out, nid

def build(tok, lang, tmpl_ids, names, nid, n, rng):
    per = max(1, n // len(tmpl_ids)); batches = []
    for t in tmpl_ids:
        rows = []
        for _ in range(per):
            io, s = rng.sample(names, 2)
            n1, n2 = (io, s) if rng.random() < 0.5 else (s, io)
            ids = tok(TEMPLATES[lang][t].format(N1=n1, N2=n2, S=s))["input_ids"]
            rows.append((ids, nid[io], nid[s]))
        lens = [len(r[0]) for r in rows]
        L = max(set(lens), key=lens.count)
        rows = [r for r in rows if len(r[0]) == L]
        batches.append(dict(
            x=torch.tensor([r[0] for r in rows]),
            io=torch.tensor([r[1] for r in rows]), s=torch.tensor([r[2] for r in rows]),
            pIO=torch.tensor([r[0].index(r[1]) for r in rows]),
            pS2=torch.tensor([len(r[0]) - 1 - r[0][::-1].index(r[2]) for r in rows])))
    return batches

def slice_b(b, n): return {k: v[:n] for k, v in b.items()}

class Hooks:
    def __init__(self): self.h = []
    def add(self, m, kind, fn): self.h.append(getattr(m, kind)(fn))
    def __enter__(self): return self
    def __exit__(self, *a): [h.remove() for h in self.h]

def subspace(X, k):
    Xc = X - X.mean(0)
    return torch.linalg.svd(Xc, full_matrices=False)[2][:k]

def angle(X, Y, k):
    s = torch.linalg.svdvals(subspace(X, k) @ subspace(Y, k).T).clamp(-1, 1)
    return torch.rad2deg(torch.arccos(s)).mean().item()

In [5]:
@torch.no_grad()
def fwd(model, b, hs_layers=(), bs=32):
    dev = model.model.embed_tokens.weight.device
    cap = getattr(model.config, "final_logit_softcapping", None)
    lds, accs = [], []
    hs = {l: {"IO": [], "S2": [], "END": []} for l in hs_layers}
    for i in range(0, len(b["x"]), bs):
        x = b["x"][i:i+bs].to(dev)
        out = model.model(input_ids=x, output_hidden_states=bool(hs_layers))
        lg = model.lm_head(out.last_hidden_state[:, -1])
        if cap: lg = torch.tanh(lg / cap) * cap
        lg = lg.float(); ar = torch.arange(len(x), device=lg.device)
        d = lg[ar, b["io"][i:i+bs].to(lg.device)] - lg[ar, b["s"][i:i+bs].to(lg.device)]
        lds.append(d.cpu()); accs.append((d > 0).float().cpu())
        for l in hs_layers:
            h = out.hidden_states[l]; a2 = torch.arange(len(x), device=h.device)
            hs[l]["IO"].append(h[a2, b["pIO"][i:i+bs].to(h.device)].float().cpu())
            hs[l]["S2"].append(h[a2, b["pS2"][i:i+bs].to(h.device)].float().cpu())
            hs[l]["END"].append(h[:, -1].float().cpu())
    return torch.cat(lds), torch.cat(accs), {l: {p: torch.cat(v) for p, v in d.items()} for l, d in hs.items()}

def ev(model, batches, hs_layers=()):
    rs = [fwd(model, b, hs_layers) for b in batches]
    ld = torch.cat([r[0] for r in rs]); acc = torch.cat([r[1] for r in rs])
    hs = {l: {p: torch.cat([r[2][l][p] for r in rs]) for p in ("IO", "S2", "END")} for l in hs_layers}
    return ld, acc, hs

def head_hook(hs_idx, HD):
    def fn(mod, args):
        x = args[0].clone()
        for h in hs_idx: x[..., h*HD:(h+1)*HD] = 0
        return (x,) + tuple(args[1:])
    return fn

def ev_heads(model, batches, heads):   # heads: list of (layer, head)
    HD = model.config.head_dim; by = {}
    for l, h in heads: by.setdefault(l, []).append(h)
    with Hooks() as H:
        for l, hh in by.items():
            H.add(model.model.layers[l].self_attn.o_proj, "register_forward_pre_hook", head_hook(hh, HD))
        return ev(model, batches)[0]

def sub_hook(V, mu):
    def fn(mod, args, out):
        tup = isinstance(out, tuple); h = (out[0] if tup else out).clone()
        last = h[:, -1].float(); Vd = V.to(h.device); m = mu.to(h.device)
        h[:, -1] = (last - ((last - m) @ Vd.T) @ Vd).to(h.dtype)
        return (h,) + tuple(out[1:]) if tup else h
    return fn

def ev_sub(model, batches, layer_idx, V, mu):
    with Hooks() as H:
        H.add(model.model.layers[layer_idx], "register_forward_hook", sub_hook(V, mu))
        return ev(model, batches)[0]

In [6]:
def build_matched(tok, names, nid, n_pairs, rng):
    """Same (io, s, order) rows in every language and every template; row i aligned everywhere."""
    pairs = [(tuple(rng.sample(names, 2)), rng.random() < 0.5) for _ in range(n_pairs)]
    data = {}
    for lang in LANGS:
        data[lang] = []
        for t in range(6):
            rows = []
            for (io, s), order in pairs:
                n1, n2 = (io, s) if order else (s, io)
                ids = tok(TEMPLATES[lang][t].format(N1=n1, N2=n2, S=s))["input_ids"]
                rows.append((ids, nid[io], nid[s]))
            L = len(rows[0][0]); assert all(len(r[0]) == L for r in rows), "length mismatch (multi-token name?)"
            data[lang].append(dict(
                x=torch.tensor([r[0] for r in rows]),
                io=torch.tensor([r[1] for r in rows]), s=torch.tensor([r[2] for r in rows]),
                pIO=torch.tensor([r[0].index(r[1]) for r in rows]),
                pS2=torch.tensor([len(r[0]) - 1 - r[0][::-1].index(r[2]) for r in rows])))
    return data

def positions(b, pos_names):
    d = {"IO": b["pIO"], "S2": b["pS2"], "END": torch.full_like(b["pIO"], b["x"].shape[1] - 1)}
    return torch.stack([d[p] for p in pos_names], 1)          # [n, P]

def by_layer(heads):
    out = {}
    for l, h in heads: out.setdefault(l, []).append(h)
    return out

@torch.no_grad()
def capture_heads(model, b, heads, pos_names, bs=32):
    """pre-o_proj head outputs at given positions: {layer: [n, P, len(heads_in_layer), HD]}"""
    HD, NH = model.config.head_dim, model.config.num_attention_heads
    BL = by_layer(heads); P = positions(b, pos_names); st = {}; out = {l: [] for l in BL}
    def mk(l):
        hh = BL[l]
        def fn(mod, args):
            x = args[0]; B = x.shape[0]
            g = x[torch.arange(B, device=x.device)[:, None], st["pos"].to(x.device)]
            out[l].append(g.view(B, g.shape[1], NH, HD)[:, :, hh].detach().clone())
        return fn
    dev = model.model.embed_tokens.weight.device
    with Hooks() as H:
        for l in BL: H.add(model.model.layers[l].self_attn.o_proj, "register_forward_pre_hook", mk(l))
        for i in range(0, len(b["x"]), bs):
            st["pos"] = P[i:i+bs]
            model.model(input_ids=b["x"][i:i+bs].to(dev))
    return {l: torch.cat(v) for l, v in out.items()}

@torch.no_grad()
def patched_ld(model, b, heads, pos_names, vals=None, bs=32):
    """logit-diff on b with heads' outputs at pos_names replaced by vals (None = zeroed)."""
    HD, NH = model.config.head_dim, model.config.num_attention_heads
    BL = by_layer(heads); P = positions(b, pos_names); st = {}
    def mk(l):
        hh = BL[l]
        def fn(mod, args):
            x = args[0].clone(); B = x.shape[0]
            ar = torch.arange(B, device=x.device)[:, None]; pos = st["pos"].to(x.device)
            xv = x.view(B, x.shape[1], NH, HD)
            cur = xv[ar, pos]
            cur[:, :, hh] = torch.zeros_like(cur[:, :, hh]) if vals is None else st["vals"][l].to(x.device)
            xv[ar, pos] = cur
            return (x,) + tuple(args[1:])
        return fn
    lds = []
    with Hooks() as H:
        for l in BL: H.add(model.model.layers[l].self_attn.o_proj, "register_forward_pre_hook", mk(l))
        for i in range(0, len(b["x"]), bs):
            st["pos"] = P[i:i+bs]
            if vals is not None: st["vals"] = {l: v[i:i+bs] for l, v in vals.items()}
            lds.append(fwd(model, {k: v[i:i+bs] for k, v in b.items()}, bs=bs)[0])
    return torch.cat(lds)

def patch_eval(model, tgt_b, src_b, heads, pos_names):
    vals = capture_heads(model, src_b, heads, pos_names)
    return patched_ld(model, tgt_b, heads, pos_names, vals)

def load_tops(tag):
    p = f"exp13_results/{tag}.json"
    if SMOKE or not os.path.exists(p): return None
    t = json.load(open(p))["heads"]["top"]
    if not all(k in t for k in ("EN", "FR", "YO")): return None
    return {k: [tuple(x) for x in t[k]] for k in ("EN", "FR", "YO")}

In [7]:
def run_model(path):
    tag = os.path.basename(path.rstrip("/"))
    out = f"{RESULTS_DIR}/{tag}{'_smoke' if SMOKE else ''}.json"
    if os.path.exists(out):
        print("cached", out); return json.load(open(out))
    tok = AutoTokenizer.from_pretrained(path, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16, device_map="auto",
                                                 attn_implementation="eager", token=HF_TOKEN).eval()
    for p in model.parameters(): p.requires_grad_(False)
    NL, NH = model.config.num_hidden_layers, model.config.num_attention_heads
    names, nid = single_token_names(tok, NAMES["en"])
    random.Random(7).shuffle(names)
    print(tag, "| EN single-token names:", len(names), "| devices:", sorted(set(map(str, model.hf_device_map.values()))) if hasattr(model, "hf_device_map") else "single")

    tops = load_tops(tag)
    if tops is None:
        print("scanning heads (no cached ranking)")
        Ds = build_matched(tok, names, nid, SCAN_PER_T, random.Random(100)); tops = {}
        for L, k in (("en", "EN"), ("fr", "FR"), ("yo", "YO")):
            sbs = Ds[L]; base = ev(model, sbs)[0].mean().item()
            sc = np.zeros((NL, NH))
            for l in range(NL):
                for h in range(NH):
                    sc[l, h] = base - ev_heads(model, sbs, [(l, h)]).mean().item()
            tops[k] = [(l, h) for _, l, h in sorted(((sc[l, h], l, h) for l in range(NL) for h in range(NH)), reverse=True)[:max(K_LIST)]]
    else:
        print("using head rankings from exp13_results")
    allh = [(l, h) for l in range(NL) for h in range(NH)]
    res = {"model": tag, "tops": {k: [list(x) for x in v] for k, v in tops.items()}, "rows": [], "gate": {}}

    for seed in SEEDS:
        D = build_matched(tok, names, nid, N_PAIRS, random.Random(seed))
        clean, ok = {}, {}
        for L in LANGS:
            ld, acc, _ = ev(model, D[L])
            clean[L] = ld; ok[L] = acc.mean().item() >= GATE_ACC
            res["gate"][f"{L}_seed{seed}"] = acc.mean().item()
        print("seed", seed, "GATE", {k: round(v, 3) for k, v in res["gate"].items() if k.endswith(f"seed{seed}")}, ok)
        for tgt in LANGS:
            if not ok[tgt] or not ok["en"]: continue
            for rank in (["EN"] if tgt == "en" else ["EN", "tgt"]):
                full = tops["EN"] if rank == "EN" else tops[tgt.upper()]
                for K in K_LIST:
                    heads = full[:K]
                    for pm, pn in POSMODES.items():
                        cat = lambda xs: torch.cat(xs)
                        abl = cat([patched_ld(model, D[tgt][t], heads, pn) for t in range(6)])
                        floor = cat([patch_eval(model, D[tgt][t], D[tgt][(t + 1) % 6], heads, pn) for t in range(6)])
                        row = {"seed": seed, "tgt": tgt, "rank": rank, "K": K, "pos": pm,
                               "clean": clean[tgt].mean().item(), "abl": abl.mean().item(), "floor": floor.mean().item()}
                        if tgt != "en":
                            row["cross"] = cat([patch_eval(model, D[tgt][t], D["en"][t], heads, pn) for t in range(6)]).mean().item()
                            rs = []
                            for j in range(3):
                                rh = random.Random(seed * 100 + j).sample(allh, K)
                                rs.append(cat([patch_eval(model, D[tgt][t], D["en"][t], rh, pn) for t in range(6)]).mean().item())
                            row["rand"] = float(np.mean(rs))
                        res["rows"].append(row)
        print("seed", seed, "done")
    json.dump(res, open(out, "w"), indent=1)
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

In [8]:
ALL = {m: run_model(m) for m in MODELS}

def compat(x, row):
    d = row["clean"] - row["abl"]
    return float("nan") if (d / row["clean"] < MIN_ABL) else (x - row["abl"]) / d

TABLE = []
for m, r in ALL.items():
    for row in r["rows"]:
        e = {"model": r["model"], "seed": row["seed"], "tgt": row["tgt"], "rank": row["rank"], "K": row["K"], "pos": row["pos"],
             "clean": row["clean"], "abl": row["abl"], "abl_frac": 1 - row["abl"] / row["clean"],
             "floor_c": compat(row["floor"], row)}
        if "cross" in row:
            e["cross_c"] = compat(row["cross"], row); e["rand_c"] = compat(row["rand"], row)
        TABLE.append(e)
df = pd.DataFrame(TABLE)
for m, r in ALL.items():
    print("\n=====", r["model"], "| gate:", {k: round(v, 3) for k, v in r["gate"].items()})
    d = df[df.model == r["model"]]
    cols = [c for c in ["clean", "abl_frac", "floor_c", "cross_c", "rand_c"] if c in d.columns]
    g = d.groupby(["tgt", "rank", "K", "pos"])[cols].agg(["mean", "std"]).round(2)
    print(g.to_string())

print("\n##### SUMMARY: compat (1 = interchangeable, 0 = as bad as ablation); heads ranked on EN, K=%d, named positions #####" % max(K_LIST))
s = df[(df["rank"] == "EN") & (df.K == max(K_LIST)) & (df.pos == "named") & (df.tgt != "en")]
if len(s):
    print(s.groupby(["model", "tgt"])[["abl_frac", "floor_c", "cross_c", "rand_c"]].mean().round(2).to_string())
print("\n(nan = ablating those heads removed < %.0f%% of the logit-diff, so compat is undefined)" % (100 * MIN_ABL))

Loading weights: 100%|██████████| 464/464 [00:02<00:00, 215.00it/s]


gemma-2-9b-it | EN single-token names: 45 | devices: single
using head rankings from exp13_results
seed 0 GATE {'en_seed0': 1.0, 'fr_seed0': 1.0, 'yo_seed0': 1.0} {'en': True, 'fr': True, 'yo': True}
seed 0 done
seed 1 GATE {'en_seed1': 1.0, 'fr_seed1': 1.0, 'yo_seed1': 0.99} {'en': True, 'fr': True, 'yo': True}
seed 1 done
seed 2 GATE {'en_seed2': 1.0, 'fr_seed2': 1.0, 'yo_seed2': 0.997} {'en': True, 'fr': True, 'yo': True}
seed 2 done


Loading weights: 100%|██████████| 508/508 [00:05<00:00, 98.81it/s] 


gemma-2-27b-it | EN single-token names: 45 | devices: single
using head rankings from exp13_results
seed 0 GATE {'en_seed0': 1.0, 'fr_seed0': 1.0, 'yo_seed0': 1.0} {'en': True, 'fr': True, 'yo': True}
seed 0 done
seed 1 GATE {'en_seed1': 0.997, 'fr_seed1': 0.997, 'yo_seed1': 1.0} {'en': True, 'fr': True, 'yo': True}
seed 1 done
seed 2 GATE {'en_seed2': 1.0, 'fr_seed2': 1.0, 'yo_seed2': 1.0} {'en': True, 'fr': True, 'yo': True}
seed 2 done


Loading weights: 100%|██████████| 288/288 [00:00<00:00, 583.73it/s]


gemma-2-2b | EN single-token names: 45 | devices: single
using head rankings from exp13_results
seed 0 GATE {'en_seed0': 1.0, 'fr_seed0': 0.972, 'yo_seed0': 0.986} {'en': True, 'fr': True, 'yo': True}
seed 0 done
seed 1 GATE {'en_seed1': 1.0, 'fr_seed1': 0.972, 'yo_seed1': 0.962} {'en': True, 'fr': True, 'yo': True}
seed 1 done
seed 2 GATE {'en_seed2': 1.0, 'fr_seed2': 0.979, 'yo_seed2': 0.983} {'en': True, 'fr': True, 'yo': True}
seed 2 done


Loading weights: 100%|██████████| 464/464 [00:01<00:00, 349.23it/s]


gemma-2-9b | EN single-token names: 45 | devices: single
using head rankings from exp13_results
seed 0 GATE {'en_seed0': 0.997, 'fr_seed0': 1.0, 'yo_seed0': 0.983} {'en': True, 'fr': True, 'yo': True}
seed 0 done
seed 1 GATE {'en_seed1': 0.997, 'fr_seed1': 1.0, 'yo_seed1': 0.99} {'en': True, 'fr': True, 'yo': True}
seed 1 done
seed 2 GATE {'en_seed2': 0.997, 'fr_seed2': 1.0, 'yo_seed2': 0.983} {'en': True, 'fr': True, 'yo': True}
seed 2 done

===== gemma-2-9b-it | gate: {'en_seed0': 1.0, 'fr_seed0': 1.0, 'yo_seed0': 1.0, 'en_seed1': 1.0, 'fr_seed1': 1.0, 'yo_seed1': 0.99, 'en_seed2': 1.0, 'fr_seed2': 1.0, 'yo_seed2': 0.997}
                  clean       abl_frac       floor_c       cross_c       rand_c      
                   mean   std     mean   std    mean   std    mean   std   mean   std
tgt rank K  pos                                                                      
en  EN   5  end    7.09  0.08     0.48  0.00    1.03  0.00     NaN   NaN    NaN   NaN
            named  7.09 